# Generate Random Parameter Samples v2

Generate 28 uncorrelated parameter combinations, then simulate responses and check localization metrics.

**Parameter ranges**:
- `sigma_spectral`: 2-20 (log-spaced)
- `sigma_prior`: 5-90 (log-spaced)
- `sigma_motor`: 5-16 (linear)

**Target metric ranges**:
- rmsL: 5-20°
- querr: 0-35%
- rmsPmedianlocal: 15-40°

The 28 combinations printed at the end are hard-coded in notebooks 03 and 04. The summary of the
simulated metrics (mean and [min, max] of LE, PE, QE) is reported in Sec. 3.3.

*Saved outputs* come from the run used for the paper (development version of `bayesian_listener`
preceding 0.2.0, KEMAR GRAS large-ears SOFA file before removal of its 35 duplicated top positions).
Re-running with 0.2.0 and `data/hrtf/KEMAR_FreeFieldCompMinPhase_48kHz.sofa` reproduces the
parameter combinations exactly; the simulated metrics change slightly (different random draws).

In [2]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import pyfar as pf
from scipy.stats import spearmanr
from pathlib import Path
from joblib import Parallel, delayed
import warnings
warnings.filterwarnings('ignore')

from bayesian_listener import BayesianListener
import bayesian_listener.metrics as metrics_module
from bayesian_listener.fitting import sigma_to_kappa

import paths


## Generate 28 uncorrelated parameter combinations

In [3]:
np.random.seed(42)
n = 28

def create_uncorrelated_samples(n, n_params, max_iter=10000):
    """Create n samples of n_params with minimal Spearman correlation."""
    best_samples = None
    best_max_corr = 1.0

    for _ in range(max_iter):
        samples = np.zeros((n, n_params))
        samples[:, 0] = np.arange(n)
        for j in range(1, n_params):
            samples[:, j] = np.random.permutation(n)

        max_corr = 0
        for i in range(n_params):
            for j in range(i+1, n_params):
                r, _ = spearmanr(samples[:, i], samples[:, j])
                max_corr = max(max_corr, abs(r))

        if max_corr < best_max_corr:
            best_max_corr = max_corr
            best_samples = samples.copy()

        if best_max_corr < 0.1:
            break

    return best_samples, best_max_corr

ranks, max_corr = create_uncorrelated_samples(n, 3, max_iter=50000)
print(f'Max absolute Spearman correlation: {max_corr:.3f}')

# Map ranks to parameter values
# sigma_spectral: 2-20 (log-spaced)
spectral_values = np.exp(np.linspace(np.log(2), np.log(15), n))
# sigma_prior: 5-90 (log-spaced)
prior_values = np.exp(np.linspace(np.log(5), np.log(90), n))
# sigma_motor: 5-16 (linear)
motor_values = np.linspace(5, 16, n)

sigma_spectral = spectral_values[ranks[:, 0].astype(int)]
sigma_prior = prior_values[ranks[:, 1].astype(int)]
sigma_motor = motor_values[ranks[:, 2].astype(int)]

# Verify correlations
print('\nSpearman correlations:')
print(f'  spectral vs prior: {spearmanr(sigma_spectral, sigma_prior)[0]:.3f}')
print(f'  spectral vs motor: {spearmanr(sigma_spectral, sigma_motor)[0]:.3f}')
print(f'  prior vs motor: {spearmanr(sigma_prior, sigma_motor)[0]:.3f}')

print(f'\nParameter ranges:')
print(f'  sigma_spectral: {sigma_spectral.min():.1f} - {sigma_spectral.max():.1f}')
print(f'  sigma_prior: {sigma_prior.min():.1f} - {sigma_prior.max():.1f}')
print(f'  sigma_motor: {sigma_motor.min():.1f} - {sigma_motor.max():.1f}')

# Build param dicts
PARAM_COMBINATIONS = []
for i in range(n):
    PARAM_COMBINATIONS.append({
        'sigma_itd': 0.569, 'sigma_ild': 1.0,
        'sigma_spectral': round(sigma_spectral[i], 1),
        'sigma_prior': round(sigma_prior[i], 1),
        'sigma_motor': round(sigma_motor[i], 1),
    })

Max absolute Spearman correlation: 0.080

Spearman correlations:
  spectral vs prior: -0.080
  spectral vs motor: -0.024
  prior vs motor: -0.040

Parameter ranges:
  sigma_spectral: 2.0 - 15.0
  sigma_prior: 5.0 - 90.0
  sigma_motor: 5.0 - 16.0


## Check localization metrics

In [4]:
# KEMAR HRTF SOFA file (SONICOM dataset), shipped with the repository
sofa_path = str(paths.KEMAR_SOFA)
CACHE_DIR = paths.CACHE_DIR
TRIALS_PER_TARGET = 6
BASE_SEED = 42

TARGET_GRID = np.array([
    [315,  30], [315, -30], [180, -30], [ 90,  30],
    [120,   0], [ 90,   0], [225,  30], [330,   0],
    [270, -30], [  0, -30], [180,  30], [135,  30],
    [270,  30], [300,   0], [ 30,   0], [240,   0],
    [ 60,   0], [225, -30], [135, -30], [  0,  90],
    [210,   0], [270,  60], [  0,   0], [  0,  30],
    [ 90,  60], [180,  60], [180,   0], [270,   0],
    [ 45,  30], [ 45, -30], [  0,  60], [150,   0],
    [ 90, -30],
])

_tg_arr = np.column_stack([
    np.deg2rad(TARGET_GRID[:, 0]),
    np.deg2rad(TARGET_GRID[:, 1]),
    np.ones(len(TARGET_GRID)),
])
target_coords = pf.Coordinates.from_spherical_elevation(_tg_arr[:, 0], _tg_arr[:, 1], _tg_arr[:, 2])

print(f'{len(TARGET_GRID)} targets, {len(TARGET_GRID) * TRIALS_PER_TARGET} trials per simulation')

33 targets, 198 trials per simulation


In [5]:
def compute_metrics_single(combo_idx, params, sofa_path, target_coords, trials_per_target, seed):
    """Simulate responses for one param combo and compute localization metrics."""
    model = BayesianListener(sofa_path)
    model.compute_template(interpolation='SHMAX', cache_dir=CACHE_DIR)

    # restrict the target features to the 33 experimental directions
    target_indices = model.target.coords.find_nearest(target_coords)[0][0]
    model.target = model.target[target_indices]
    target_sph = target_coords.spherical_elevation

    kappa_motor = sigma_to_kappa(params['sigma_motor'])
    model.parameters = {k: v for k, v in params.items() if k != 'sigma_motor'} | {'kappa_motor': kappa_motor}
    posterior = model.infer(repetitions=trials_per_target, seed=seed)
    estimations = model.estimate(posterior, kappa_motor=kappa_motor, seed=seed).cartesian

    rows = []
    for t_idx in range(len(target_indices)):
        resp_cart = estimations[t_idx, :, :]
        resp_cart = resp_cart / np.linalg.norm(resp_cart, axis=1, keepdims=True)
        resp_coords = pf.Coordinates.from_cartesian(resp_cart[:, 0], resp_cart[:, 1], resp_cart[:, 2])
        resp_sph = resp_coords.spherical_elevation
        for i in range(trials_per_target):
            rows.append({
                'azi_target': np.rad2deg(target_sph[t_idx, 0]),
                'ele_target': np.rad2deg(target_sph[t_idx, 1]),
                'azi_response': np.rad2deg(resp_sph[i, 0]),
                'ele_response': np.rad2deg(resp_sph[i, 1]),
            })

    obs_df = pd.DataFrame(rows)
    targets_hp = target_coords.spherical_side

    _resp_arr = np.column_stack([
        np.deg2rad(obs_df['azi_response'].values),
        np.deg2rad(obs_df['ele_response'].values),
        np.ones(len(obs_df))
    ])
    resp_coords = pf.Coordinates.from_spherical_elevation(_resp_arr[:, 0], _resp_arr[:, 1], _resp_arr[:, 2])
    resp_hp = resp_coords.spherical_side

    _targ_arr = np.column_stack([
        np.deg2rad(obs_df['azi_target'].values),
        np.deg2rad(obs_df['ele_target'].values),
        np.ones(len(obs_df))
    ])
    targ_coords = pf.Coordinates.from_spherical_elevation(_targ_arr[:, 0], _targ_arr[:, 1], _targ_arr[:, 2])
    targ_hp = targets_hp[target_coords.find_nearest(targ_coords)[0][0], :]

    rmsL, _ = metrics_module.METRIC_FUNCTIONS['rmsL'](targ_hp, resp_hp)
    rmsPlocal, _ = metrics_module.METRIC_FUNCTIONS['rmsPmedianlocal'](targ_hp, resp_hp)
    querr, _ = metrics_module.METRIC_FUNCTIONS['querrMiddlebrooks'](targ_hp, resp_hp)

    return {
        'combo_idx': combo_idx,
        'sigma_spectral': params['sigma_spectral'],
        'sigma_prior': params['sigma_prior'],
        'sigma_motor': params['sigma_motor'],
        'rmsL_deg': np.rad2deg(rmsL),
        'rmsPlocal_deg': np.rad2deg(rmsPlocal),
        'querr': querr,
    }

In [6]:
import time

print(f'Computing metrics for {len(PARAM_COMBINATIONS)} combinations...')
t_start = time.time()

results = Parallel(n_jobs=-1, verbose=5)(
    delayed(compute_metrics_single)(
        i, params, sofa_path, target_coords, TRIALS_PER_TARGET, BASE_SEED + i * 100
    )
    for i, params in enumerate(PARAM_COMBINATIONS)
)

metrics_df = pd.DataFrame(results)
print(f'Done in {time.time() - t_start:.0f}s')

Computing metrics for 28 combinations...


[Parallel(n_jobs=-1)]: Using backend LokyBackend with 14 concurrent workers.


✓ Loading from cache: KEMAR_GRAS_EarSim_LargeEars_FreeFieldCompMinPhase_48kHz_551d850c_SHMAX_20260211_182611.pkl
✓ Cache loaded successfully
✓ Loading from cache: KEMAR_GRAS_EarSim_LargeEars_FreeFieldCompMinPhase_48kHz_551d850c_SHMAX_20260211_182611.pkl
✓ Cache loaded successfully
✓ Loading from cache: KEMAR_GRAS_EarSim_LargeEars_FreeFieldCompMinPhase_48kHz_551d850c_SHMAX_20260211_182611.pkl
✓ Cache loaded successfully
✓ Loading from cache: KEMAR_GRAS_EarSim_LargeEars_FreeFieldCompMinPhase_48kHz_551d850c_SHMAX_20260211_182611.pkl
✓ Cache loaded successfully
✓ Loading from cache: KEMAR_GRAS_EarSim_LargeEars_FreeFieldCompMinPhase_48kHz_551d850c_SHMAX_20260211_182611.pkl
✓ Loading from cache: KEMAR_GRAS_EarSim_LargeEars_FreeFieldCompMinPhase_48kHz_551d850c_SHMAX_20260211_182611.pkl
✓ Loading from cache: KEMAR_GRAS_EarSim_LargeEars_FreeFieldCompMinPhase_48kHz_551d850c_SHMAX_20260211_182611.pkl
✓ Cache loaded successfully
✓ Cache loaded successfully
✓ Cache loaded successfully
✓ Loading fro

OMP: Info #276: omp_set_nested routine deprecated, please use omp_set_max_active_levels instead.
OMP: Info #276: omp_set_nested routine deprecated, please use omp_set_max_active_levels instead.
OMP: Info #276: omp_set_nested routine deprecated, please use omp_set_max_active_levels instead.
OMP: Info #276: omp_set_nested routine deprecated, please use omp_set_max_active_levels instead.
OMP: Info #276: omp_set_nested routine deprecated, please use omp_set_max_active_levels instead.
OMP: Info #276: omp_set_nested routine deprecated, please use omp_set_max_active_levels instead.
OMP: Info #276: omp_set_nested routine deprecated, please use omp_set_max_active_levels instead.
OMP: Info #276: omp_set_nested routine deprecated, please use omp_set_max_active_levels instead.
OMP: Info #276: omp_set_nested routine deprecated, please use omp_set_max_active_levels instead.
OMP: Info #276: omp_set_nested routine deprecated, please use omp_set_max_active_levels instead.
OMP: Info #276: omp_set_nested

✓ Loading from cache: KEMAR_GRAS_EarSim_LargeEars_FreeFieldCompMinPhase_48kHz_551d850c_SHMAX_20260211_182611.pkl
✓ Cache loaded successfully
✓ Loading from cache: KEMAR_GRAS_EarSim_LargeEars_FreeFieldCompMinPhase_48kHz_551d850c_SHMAX_20260211_182611.pkl
✓ Cache loaded successfully
✓ Loading from cache: KEMAR_GRAS_EarSim_LargeEars_FreeFieldCompMinPhase_48kHz_551d850c_SHMAX_20260211_182611.pkl
✓ Cache loaded successfully
✓ Loading from cache: KEMAR_GRAS_EarSim_LargeEars_FreeFieldCompMinPhase_48kHz_551d850c_SHMAX_20260211_182611.pkl
✓ Cache loaded successfully
✓ Loading from cache: KEMAR_GRAS_EarSim_LargeEars_FreeFieldCompMinPhase_48kHz_551d850c_SHMAX_20260211_182611.pkl
✓ Loading from cache: KEMAR_GRAS_EarSim_LargeEars_FreeFieldCompMinPhase_48kHz_551d850c_SHMAX_20260211_182611.pkl
✓ Cache loaded successfully
✓ Cache loaded successfully
✓ Loading from cache: KEMAR_GRAS_EarSim_LargeEars_FreeFieldCompMinPhase_48kHz_551d850c_SHMAX_20260211_182611.pkl
✓ Loading from cache: KEMAR_GRAS_EarSim_L

[Parallel(n_jobs=-1)]: Done  19 out of  28 | elapsed:    4.6s remaining:    2.2s
[Parallel(n_jobs=-1)]: Done  25 out of  28 | elapsed:    4.6s remaining:    0.6s
[Parallel(n_jobs=-1)]: Done  28 out of  28 | elapsed:    4.7s finished


## Results

In [7]:
# Print all combinations with metrics
print(metrics_df[['combo_idx', 'sigma_spectral', 'sigma_prior', 'sigma_motor',
                   'rmsL_deg', 'rmsPlocal_deg', 'querr']].to_string(
    index=False, float_format='%.1f'))


 combo_idx  sigma_spectral  sigma_prior  sigma_motor  rmsL_deg  rmsPlocal_deg  querr
         0             2.0         22.4          7.9       8.7           16.3    0.0
         1             2.2          6.9         14.0      14.2           35.4    3.8
         2             2.3         42.5          5.0       5.4           10.8    0.0
         3             2.5         58.7         15.2      14.5           18.0    0.0
         4             2.7         18.1         13.1      11.5           21.6    2.4
         5             2.9         13.1         16.0      16.8           25.9    2.4
         6             3.1         20.1          5.4       6.4           28.2    1.3
         7             3.4         65.3          9.5       8.6           17.1    4.4
         8             3.6         30.9          7.0       7.7           21.6    4.1
         9             3.9         34.3          5.8       7.7           27.2    2.6
        10             4.2         38.2         14.4      14.1   

In [9]:
print(metrics_df[['sigma_spectral', 'sigma_prior', 'sigma_motor']].describe())

print(metrics_df[['rmsL_deg', 'rmsPlocal_deg', 'querr']].describe())




       sigma_spectral  sigma_prior  sigma_motor
count       28.000000    28.000000    28.000000
mean         6.525000    30.085714    10.500000
std          3.870269    24.788538     3.356696
min          2.000000     5.000000     5.000000
25%          3.325000    10.325000     7.775000
50%          5.500000    21.250000    10.500000
75%          9.075000    43.700000    13.225000
max         15.000000    90.000000    16.000000
        rmsL_deg  rmsPlocal_deg      querr
count  28.000000      28.000000  28.000000
mean   10.922720      31.126974  12.417413
std     3.021578       8.774332   9.347354
min     5.422098      10.767115   0.000000
25%     8.714259      25.888355   3.542510
50%    11.026892      33.456738  13.459836
75%    13.632072      38.344808  20.566267
max    16.764698      42.971515  27.160494


## Output

In [15]:
# Print copy-pasteable PARAM_COMBINATIONS
output = metrics_df.sort_values('sigma_spectral').reset_index(drop=True)

print('PARAM_COMBINATIONS = [')
for _, row in output.iterrows():
    print(f"    {{'sigma_itd': 0.569, 'sigma_ild': 1.0, "
          f"'sigma_spectral': {row['sigma_spectral']:.1f}, "
          f"'sigma_prior': {row['sigma_prior']:.1f}, "
          f"'sigma_motor': {row['sigma_motor']:.1f}}},"
          f"  # rmsP={row['rmsPlocal_deg']:.1f} rmsL={row['rmsL_deg']:.1f} qe={row['querr']:.1f}")
print(']')

PARAM_COMBINATIONS = [
    {'sigma_itd': 0.569, 'sigma_ild': 1.0, 'sigma_spectral': 2.0, 'sigma_prior': 22.4, 'sigma_motor': 7.9},  # rmsP=14.2 rmsL=8.1 qe=0.0
    {'sigma_itd': 0.569, 'sigma_ild': 1.0, 'sigma_spectral': 2.2, 'sigma_prior': 6.9, 'sigma_motor': 14.0},  # rmsP=32.6 rmsL=13.4 qe=5.9
    {'sigma_itd': 0.569, 'sigma_ild': 1.0, 'sigma_spectral': 2.3, 'sigma_prior': 42.5, 'sigma_motor': 5.0},  # rmsP=10.9 rmsL=5.5 qe=0.0
    {'sigma_itd': 0.569, 'sigma_ild': 1.0, 'sigma_spectral': 2.5, 'sigma_prior': 58.7, 'sigma_motor': 15.2},  # rmsP=19.6 rmsL=15.5 qe=0.0
    {'sigma_itd': 0.569, 'sigma_ild': 1.0, 'sigma_spectral': 2.7, 'sigma_prior': 18.1, 'sigma_motor': 13.1},  # rmsP=22.9 rmsL=12.3 qe=1.3
    {'sigma_itd': 0.569, 'sigma_ild': 1.0, 'sigma_spectral': 2.9, 'sigma_prior': 13.1, 'sigma_motor': 16.0},  # rmsP=26.5 rmsL=15.9 qe=2.6
    {'sigma_itd': 0.569, 'sigma_ild': 1.0, 'sigma_spectral': 3.1, 'sigma_prior': 20.1, 'sigma_motor': 5.4},  # rmsP=26.8 rmsL=6.4 qe=2.7
    {'sigma